In [1]:
!pip install datasets transformers torch

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import json

In [2]:
import json, os

if os.path.exists("donnees_propres.json"):
    chemin = "donnees_propres.json"
elif os.path.exists("notebooks/donnees_propres.json"):
    chemin = "notebooks/donnees_propres.json"
else:
    raise FileNotFoundError("❌ donnees_propres.json introuvable !")

with open(chemin, "r") as f:
    donnees = json.load(f)

print(f"✅ {len(donnees)} exemples chargés depuis : {chemin}")

print(f"Nombre d'exemples chargés : {len(donnees)}")
print(f"Exemple 1 — nombre de chunks : {donnees[0]['nb_chunks']}")
print(f"Résumé : {donnees[0]['resume'][:200]}")

✅ 500 exemples chargés depuis : donnees_propres.json
Nombre d'exemples chargés : 500
Exemple 1 — nombre de chunks : 16
Résumé : Before any characters appear, the time and geography are made clear. Though it is the last war that England and France waged for a country that neither would retain, the wilderness between the forces 


In [3]:
def chunking_avec_overlap(texte, taille=512, overlap=50):
    """
    taille  = nombre de mots par chunk
    overlap = nombre de mots qui se répètent entre deux chunks
    """
    mots = texte.split()
    chunks = []
    i = 0
    while i < len(mots):
        # Prendre un chunk de 'taille' mots
        chunk = mots[i : i + taille]
        chunks.append(' '.join(chunk))
        # Avancer de (taille - overlap) pour créer le chevauchement
        i += taille - overlap
    return chunks

# Tester sur le premier exemple
texte_test = donnees[0]['chunks'][0]  # premier chunk du premier livre
chunks_overlap = chunking_avec_overlap(texte_test, taille=128, overlap=20)

print(f"Nombre de chunks avec overlap : {len(chunks_overlap)}")
print(f"\nFin du chunk 1 : ...{chunks_overlap[0][-100:]}")
print(f"\nDébut du chunk 2 : {chunks_overlap[1][:100]}...")
print("\n✅ Tu vois que les deux chunks partagent des mots en commun !")

Nombre de chunks avec overlap : 5

Fin du chunk 1 : ...rage in a more martial conflict. But, emulating the patience and self-denial of the practised native

Début du chunk 2 : opportunity to exhibit their courage in a more martial conflict. But, emulating the patience and sel...

✅ Tu vois que les deux chunks partagent des mots en commun !


In [ ]:
import json, os
from transformers import AutoTokenizer

# Vérifier si le fichier existe
if os.path.exists("tokenizer_choisi.json"):
    with open("tokenizer_choisi.json", "r", encoding="latin-1") as f:
        decision = json.load(f)
    modele = decision["modele_huggingface"]
else:
    # Fallback si ta collègue n'a pas encore généré le fichier
    modele = "google/pegasus-xsum"
    print("⚠️ tokenizer_choisi.json introuvable → fallback PEGASUS")

tokenizer = AutoTokenizer.from_pretrained(modele)
print(f"✅ Tokenizer chargé : {modele}")

# Tester la tokenisation d'un chunk
chunk_exemple = chunks_overlap[0]
tokens = tokenizer(
    chunk_exemple,
    max_length=512,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

print("Forme des input_ids :", tokens["input_ids"].shape)
print("Premier token :", tokens["input_ids"][0][:10])

IndentationError: expected an indented block after 'if' statement on line 5 (775374017.py, line 6)

In [ ]:
class BookSumDataset(Dataset):
    def __init__(self, donnees, tokenizer, max_length=512):
        self.donnees = donnees
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # Combien d'exemples on a
        return len(self.donnees)

    def __getitem__(self, idx):
        exemple = self.donnees[idx]

        # Prendre le texte complet (tous les chunks joints)
        texte = ' '.join(exemple['chunks'])
        resume = exemple['resume']

        # Tokeniser le texte (entrée)
        entree = self.tokenizer(
            texte,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # Tokeniser le résumé (sortie attendue)
        sortie = self.tokenizer(
            resume,
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": entree["input_ids"].squeeze(),
            "attention_mask": entree["attention_mask"].squeeze(),
            "labels": sortie["input_ids"].squeeze()
        }

print("✅ Classe BookSumDataset définie !")

✅ Classe BookSumDataset définie !


In [ ]:
# Créer le dataset
train_dataset = BookSumDataset(donnees[:80], tokenizer)
val_dataset   = BookSumDataset(donnees[80:], tokenizer)

# Créer les DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

print(f"Taille train dataset : {len(train_dataset)} exemples")
print(f"Taille val dataset   : {len(val_dataset)} exemples")
print(f"Nombre de batches train : {len(train_loader)}")

# Tester un batch
batch = next(iter(train_loader))
print(f"\nForme d'un batch :")
print(f"  input_ids    : {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  labels       : {batch['labels'].shape}")

Taille train dataset : 80 exemples
Taille val dataset   : 20 exemples
Nombre de batches train : 20

Forme d'un batch :
  input_ids    : torch.Size([4, 512])
  attention_mask: torch.Size([4, 512])
  labels       : torch.Size([4, 128])


In [ ]:
# Sauvegarder le dataset préparé
torch.save(train_dataset, "train_dataset.pt")
torch.save(val_dataset,   "val_dataset.pt")

print("✅ DataLoader prêt !")
print("✅ train_dataset.pt sauvegardé")
print("✅ val_dataset.pt sauvegardé")
print("\nTa camarade peut maintenant utiliser ces datasets pour le fine-tuning !")

✅ DataLoader prêt !
✅ train_dataset.pt sauvegardé
✅ val_dataset.pt sauvegardé

Ta camarade peut maintenant utiliser ces datasets pour le fine-tuning !
